In [1]:
import multiprocessing
import os
import re
import pandas as pd
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain, SequentialChain
from langchain_community.chat_models import ChatLlamaCpp

/Users/isabel/anaconda3/envs/syntheticDataGeneration/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/isabel/anaconda3/envs/syntheticDataGeneration/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Silence llama.cpp
os.environ["LLAMA_LOG_LEVEL"] = "ERROR" 

In [3]:
def generate_notes(system_role_prompt, note_query_prompt, input_data, temperature, model, completions, filename, dataset):
  # generate and save notes

    def parse_and_clean_reports(results):
        # parse and clean reports from openai completions
        reports = []

        for result in results:
            text = result.get("output", "")
            text = text.replace('\n', ' ')
            pattern = r'\*\*\*|\s(?=\d{1,2}[\.,]{1,2}\s)' # split by *** or numbers followed by . or , 
            # Split by *** or quotes
            splits = re.split(pattern, text)
            for item in splits:
                if item:
                    print(item)
                    cleaned = re.sub(r'^\s*\d{1,2}[\.,]{1,3}\s*', '', item)
                    cleaned = cleaned.strip()
                    # Keep only items with letters, no colon or quote
                    if cleaned and re.search(r'[a-zA-Z]', cleaned):  # keep only if contains letters
                        reports.append(cleaned)

        return reports

    def save_reports(reports, needs, filename):
        df = pd.DataFrame(reports, columns=['report'])
        df['needs'] = needs
        try:
            df.to_csv(filename, index=False)
            print(f"Reports saved successfully to {filename}")
        except Exception as e:
            print(f"Failed to save reports: {str(e)}")

    data_dir = f'./data/{dataset}'  # update if needed
    os.makedirs(data_dir, exist_ok=True)
    report_filepath = os.path.join(data_dir, f'{filename}_{input_data["needs"]}.csv')

    # initialize local model
    llm = ChatLlamaCpp(
    model_path=model,
    temperature=temperature,
    n_ctx=10000,
    n_gpu_layers=8,
    n_batch=300,
    max_tokens=512,
    n_threads=max(1, multiprocessing.cpu_count() - 1),
    repeat_penalty=1.5,
    top_p=0.5,
    verbose=False,
    )

    # create prompts
    role_prompt = PromptTemplate(template=system_role_prompt['message'], input_variables=system_role_prompt['inputs'])
    note_prompt = PromptTemplate(template=note_query_prompt['message'], input_variables=note_query_prompt['inputs'])

    # create llmchains
    role_chain = LLMChain(llm=llm, prompt=role_prompt, output_key = "intermediate_output")
    note_chain = LLMChain(llm=llm, prompt=note_prompt, output_key = "output")
    # create sequentialchain
    sequential_chain = SequentialChain(
        chains=[role_chain, note_chain], input_variables = (system_role_prompt['inputs'] + note_query_prompt['inputs']), output_variables = ["output"]
    )

    # generate a number of different responses
    results = []
    for _ in range(completions):
        response = sequential_chain.invoke(input_data)
        results.append(response)

    # clean results
    results = parse_and_clean_reports(results)
    # save results
    save_reports(results, input_data["needs"], report_filepath)


In [4]:
system_role_prompt = {'message':
                      '''
                      You are a specialist in generating fictitious data for natural language processing projects in healthcare.
                      You speak the language of a nurse in an {nationality} nursing home. Namely, you speak {language}.
                      ''' ,
                      'inputs':["nationality", "language"]}

note_query_prompt = {'message':
                      '''
                      This is an example of a nurse note for a patient in a day: "{example_note}"

                      Other reports may include: washing, dressing, brushing teeth, getting ready for the day, getting ready for the night, showering, cleaning dental prostheses, or assistance after incontinence.
                      Other reports could include: what the client has or has not eaten, what help is needed with eating (full help, encouragement, adapted cutlery or cup), choking, keeping hydration and nutrition lists.
                      Other reports could include: Organised activities, getting visitors, browsing through a magazine, interacting with fellow residents. Keep in mind that these are reports from people in a nursing home, with severe disabilities, so social interaction and activities are limited. Usually it involves sociability, but not always.
                      Other reports may include, for example: oedema, pressure ulcers, peeling, redness and itching of the skin. Nails that are too long, blemishes.
                      Other reports could include, for example: care plan discussions, minor medical complaints, family requests, ordering medication.
                      Reports can be, for example, about: restlessness and wandering at night, sleeping well, going to the toilet at night, phoning, lying crookedly in bed.
                      Reports may include: agitation, restlessness, apathy, confusion; usually the confusion is subtle, but sometimes more intense.
                      Reports may include, for example: pain, tightness of breath, nausea, diarrhoea, back pain, palliative care; usually the complaints are subtle, but sometimes more severe.
                      Other reports can be about, for example: walking aids, the wheelchair, falls, fall incidents, transfers, lifts.
                      Most reports are about everyday things, so not everything is a serious incident.

                      Make up {number_of_reports} such reports for {number_of_reports} residents with {needs} palliative care needs. Return only the reports, with each report separated by "***" and nothing else. Vary the sentence structure and style.
                      ''' ,
                      'inputs':["example_note", "number_of_reports", "needs"]}

In [5]:
dataframe = pd.read_excel(f'../fake_notes.xlsx')
completions = 25
model = './models/Phi-4-mini-instruct.Q8_0.gguf'
temperature = 1.1

for index, row in dataframe.iterrows():
  input_data = {'nationality': 'Irish', 'language': 'Hiberno-English', 'example_note': row.Note, 'number_of_reports': 25, 'needs': row.Needs}
  generate_notes(system_role_prompt, note_query_prompt, input_data, temperature, model, completions, index % 5, 'syntheticNurseNotes')

llama_context: n_ctx_seq (10240) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
/var/folders/12/p5c0vdcj3yx9xb69fcd0n3j80000gn/T/ipykernel_18527/1853770054.py:57: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  role_chain = LLMChain(llm=llm, prompt=role_prompt, output_key = "intermediate_output")


"Mary assisted John with his morning shower, noting that he seemed more comfortable today. He enjoyed a hearty lunch of soup and chicken salad in the canteen but only had half an ice cream bowl due to feeling full earlier."  
  "He helped Peter wash up after breakfast; noted some mild redness on John's skin around old pressure ulcers which needs monitoring for infection signs."   "Jane was very active this morning, enjoying her time with visitors while browsing through a magazine. She seemed quite happy and engaged in conversation today despite being under palliative care due to lung cancer."  
  "Liam had trouble swallowing his breakfast but managed half of the soup before stopping; he needed help from Mary for encouragement during meals."   "Sarah's sister visited this morning, bringing some much-needed joy as Sarah smiled brightly while watching her favorite TV show. She also mentioned feeling a bit more tired lately though still enjoying life."  
  "Oedema in John's feet was noted 

llama_context: n_ctx_seq (10240) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


KeyboardInterrupt: 